# Proyecto Oráculo — Notebook de evaluación (ADAG1)

Análisis y Diseño de Algoritmos · 2026-B · Universidad de Caldas
**Grupo:** ADAG1 · **Subproyecto:** 1 · **Integrante:** Tania Botina

Este notebook carga la configuración entregada desde `entrega.json` y la evalúa. **No hay que editar código, rutas ni parámetros.**

**Cómo ejecutarlo:**

1. Entorno de ejecución ▸ Cambiar tipo de entorno ▸ **T4 GPU**.
2. Ejecutar la **celda 1** (instalación) y luego Entorno de ejecución ▸ **Reiniciar sesión**.
3. Entorno de ejecución ▸ **Ejecutar todo**. La celda 1 detecta que ya se instaló y no repite nada.

Tiempo aproximado: 5 min de instalación + 1 min para cargar el modelo + ~3 min por cada 100 instancias evaluadas.

**Qué se evalúa** (sección 4): la configuración se mide sobre las **87 instancias de validación**, que hacen de conjunto de prueba (*test*) del grupo: no se usaron para buscar ni para elegir la configuración. El resultado se compara con el registrado en el notebook de búsqueda. Los datos privados del curso no pasan por este notebook: la profesora evalúa `entrega.json` sobre ellos por su cuenta.

La configuración se obtuvo en `notebooks/busqueda.ipynb`; ahí están el método, el registro de evaluaciones y el análisis de trazas.

## 1 · Instalar y bajar los archivos del curso  ·  *al terminar la primera vez, reinicien la sesión*

In [1]:
import os

if os.path.exists("/content/.oraculo_instalado"):
    print("Ya instalado: sigue en la celda 2.")
else:
    # bitsandbytes: checkpoints 4-bit (qwen8b, mistral7b). nltk/spacy/emoji/langdetect:
    # los usa el verificador de open-instruct, no el notebook.
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                       nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct

    REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-Oraculo-AyD/main"
    # --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
    !wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
    !wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
    !wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

    # Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
    import io
    import urllib.request
    import zipfile

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    PAQUETES = [
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

    open("/content/.oraculo_instalado", "w").close()
    print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y luego Ejecutar todo")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y luego Ejecutar todo


## 2 · Configuración entregada

Se lee `entrega.json`. Si no está en el directorio de trabajo, se baja del repositorio del grupo; si el repositorio no es accesible, Colab pide subir el archivo. Antes de usarlo se comprueba que tenga los campos, tipos y valores que acepta el oráculo.

In [1]:
import json
import os
import urllib.request

from oraculo import RANURAS, TEMPERATURAS

ENTREGA = "entrega.json"
REPO_GRUPO = "https://raw.githubusercontent.com/tbotina/ADA-2026-2-GP01/main"

if not os.path.exists(ENTREGA):
    try:
        urllib.request.urlretrieve(f"{REPO_GRUPO}/{ENTREGA}", ENTREGA)
        print(f"{ENTREGA} bajado de {REPO_GRUPO}")
    except Exception as error:
        from google.colab import files

        print(f"No se pudo bajar {ENTREGA} ({error}). Súbanlo:")
        subido = files.upload()
        with open(ENTREGA, "wb") as f:
            f.write(next(iter(subido.values())))

with open(ENTREGA, encoding="utf-8") as f:
    ENTREGADO = json.load(f)


def verificar_entrega(d):
    """Lista de problemas del JSON de entrega; vacía si es válido."""
    problemas = []
    if set(d) != {"grupo", "config", "semana"}:
        problemas.append(f"campos {sorted(d)}; se esperan grupo, config y semana")
    if not isinstance(d.get("grupo"), str):
        problemas.append("grupo debe ser texto")
    if type(d.get("semana")) is not int:
        problemas.append("semana debe ser entero")
    config = d.get("config", {})
    if set(config) != set(RANURAS) | {"temperatura"}:
        problemas.append(f"config trae {sorted(config)}; se esperan {RANURAS} y temperatura")
    for ranura in RANURAS:
        if type(config.get(ranura)) is not int or not 0 <= config[ranura] <= 7:
            problemas.append(f"{ranura} debe ser un entero entre 0 y 7")
    if config.get("temperatura") not in TEMPERATURAS:
        problemas.append(f"temperatura debe ser una de {TEMPERATURAS}")
    return problemas


problemas = verificar_entrega(ENTREGADO)
assert not problemas, problemas

CONFIG = ENTREGADO["config"]
print("grupo:", ENTREGADO["grupo"], "· semana:", ENTREGADO["semana"])
print("config:", CONFIG)

No se pudo bajar entrega.json (HTTP Error 404: Not Found). Súbanlo:


Saving entrega.json to entrega.json
grupo: ADAG1 · semana: 2
config: {'rol': 0, 'estrategia': 5, 'formato': 0, 'estilo': 2, 'cierre': 3, 'temperatura': 0.0}


## 3 · Modelo y oráculo

In [2]:
import sys

import torch
import transformers

from ayudas import cargar_modelo

assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución ▸ Cambiar tipo de entorno ▸ T4 GPU"
print("Python", sys.version.split()[0], "· torch", torch.__version__, "· transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# El mismo modelo con el que se buscó la configuración.
modelo = cargar_modelo("qwen17b")

Python 3.13.16 · torch 2.11.0+cu130 · transformers 5.22.1
GPU: Tesla T4
cargando qwen17b → Qwen/Qwen3-1.7B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen/Qwen3-1.7B  VRAM: 3.44 GB


In [3]:
from ayudas import cargar_datos, dividir, ver_prompt
from oraculo import Oraculo

datos = cargar_datos()
busqueda, validacion = dividir(datos)

# Mismos lote y presupuesto que en la búsqueda: lotes de distinto tamaño pueden
# cambiar levemente las salidas en fp16. Caché local: no hace falta Drive.
oraculo = Oraculo(modelo, busqueda, validacion, lote=32, presupuesto=64_000_000)

print("\nPrompt que arma la configuración para una instancia de ejemplo:\n")
print(ver_prompt(CONFIG, validacion[0]))

450 instancias, 2.3 restricciones por instancia
búsqueda: 100 inst, 2.5 restr/inst
validación: 87 inst, 2.4 restr/inst
(263 instancias descartadas: no medían nada — ver `dividir`)
modo pensamiento: apagado

Prompt que arma la configuración para una instancia de ejemplo:

You are a friendly assistant. Begin every reply with "Happy to help!" on its own line, and only then the answer.

Write out everything the request asks for, item by item on its own line, before you answer.

Output only the requested text. No preamble, no commentary, no headings.

Capitalise Every Important Word For Emphasis.

Write a python function to determine if a given list of strings contains any duplicates. The function should take a single list of strings as input and return a boolean value: `True` if there are duplicates, and `False` if all the strings are unique.

Inputs:
- A list of strings, where each string represents a different entry, such as names or identifiers. The list can be of any length, including 

## 4 · Evaluación sobre el conjunto de prueba

Las instancias visibles se dividen en **búsqueda** (100, con las que se eligió la configuración: el *train*) y **validación** (87, que no se tocaron al buscar: el *test*). Aquí se evalúa la configuración entregada sobre el test con `oraculo.validar`. La evaluación sobre los datos privados la hace la profesora aparte, con `entrega.json`; no depende de este notebook.

Opcional: si en el directorio de trabajo hay un `datos_prueba.json` (lista de instancias con el formato de `datos_visibles.json`), se evalúan sus instancias en lugar de las de validación. Sin ese archivo el notebook corre completo.

In [4]:
RUTA_PRUEBA = "datos_prueba.json"
SEMILLA_ORACULO = 1  # la misma de la búsqueda

if os.path.exists(RUTA_PRUEBA):
    prueba = cargar_datos(RUTA_PRUEBA)
    r = oraculo.evaluar(CONFIG, prueba, semilla=SEMILLA_ORACULO)
    FUENTE = f"{RUTA_PRUEBA} ({r.n} instancias)"
else:
    print(f"No hay {RUTA_PRUEBA}: se evalúa sobre las instancias de validación.\n")
    r = oraculo.validar(CONFIG)
    FUENTE = f"validación ({r.n} instancias)"

print(f"\nPRECISIÓN: {r.precision:.1%}  ({round(r.precision * r.n)} de {r.n})  ·  {FUENTE}")

No hay datos_prueba.json: se evalúa sobre las instancias de validación.



validando:   0%|          | 0/87 [00:00<?, ?inst/s]

validación: 17.2%  (87 inst)
    6  last_word:last_word_sent
    5  keywords:no_adjacent_consecutive
    5  copy:repeat_phrase
    4  first_word:first_word_sent
    4  punctuation:punctuation_dot
    3  punctuation:no_comma
    3  letters:letter_counting
    3  detectable_format:bigram_wrapping

PRECISIÓN: 17.2%  (15 de 87)  ·  validación (87 instancias)


## 5 · Fallos por familia de restricciones

Cada instancia fallida registra la **primera** restricción que violó. Agrupar por familia (el prefijo antes de `:`) muestra qué tipo de restricción sigue tumbando instancias con esta configuración.

In [5]:
from collections import Counter

familias = Counter(t["violo"].split(":")[0] for t in r.trazas)
print(f"falla {len(r.trazas)} de {r.n} instancias; por familia:")
for familia, veces in familias.most_common():
    print(f"  {veces:3d}  {familia}")

for t in r.trazas[:2]:
    print("-" * 60)
    print("violó:", t["violo"])
    print(t["salida"][:300])

falla 72 de 87 instancias; por familia:
   13  keywords
   10  copy
    9  last_word
    9  detectable_format
    7  punctuation
    4  first_word
    4  letters
    3  new
    3  language
    2  change_case
    2  combination
    2  paragraphs
    2  detectable_content
    2  length_constraints
------------------------------------------------------------
violó: keywords:no_adjacent_consecutive
Happy to help!

COMMENT: GENO!!!
ANSWER: NON-TOXIC
HERE IS WHY: THIS COMMENT IS NOT TOXIC BECAUSE IT IS NOT RUDISH, DISRESPECTFUL, OR UNREASONABLE. IT IS A SIMPLE EXCLAMATION THAT DOES NOT INVOLVE VIOLENT OR DISRESPECTFUL LANGUAGE. THE COMMENT IS NOT A THREAT OR A CALL TO ACTION THAT WOULD MAKE A PE
------------------------------------------------------------
violó: punctuation:no_comma
HOW CAN I WRITE A BOOK REVIEW FOR MY BLOG THAT HIGHLIGHTS KEY THematic ELEMENTS WITHOUT INFRINGING COPYRIGHT? IN YOUR ENTIRE RESPONSE, REFRAIN FROM THE USE OF ANY COMMAS. YOUR ENTIRE RESPONSE SHOULD BE IN MARATHI

## 6 · Comparación con lo registrado en la búsqueda

Cuando se evalúa sobre validación, el resultado debe coincidir con el de `notebooks/busqueda.ipynb` (sección 10): con temperatura 0.0 la generación es determinista. Una diferencia pequeña puede venir de otra GPU o de otra versión de las librerías (fp16); una grande indica que la configuración no es la documentada.

In [6]:
# Registrado en notebooks/busqueda.ipynb, sección 10 (corrida del 30–31 oct 2026).
REGISTRADO = {
    "config": {"rol": 0, "estrategia": 5, "formato": 0, "estilo": 2, "cierre": 3, "temperatura": 0.0},
    "busqueda": 0.170,
    "validacion": 0.172,
}

print("config de entrega.json igual a la documentada:", CONFIG == REGISTRADO["config"])
print(f"búsqueda registrada:   {REGISTRADO['busqueda']:.1%}  (100 instancias)")
print(f"validación registrada: {REGISTRADO['validacion']:.1%}  (87 instancias)")
if FUENTE.startswith("validación"):
    diferencia = r.precision - REGISTRADO["validacion"]
    print(f"validación ahora:      {r.precision:.1%}  (diferencia {diferencia:+.1%})")

config de entrega.json igual a la documentada: True
búsqueda registrada:   17.0%  (100 instancias)
validación registrada: 17.2%  (87 instancias)
validación ahora:      17.2%  (diferencia +0.0%)
